In [21]:
import google.generativeai as genai 
import openai
import time
import os
from dotenv import load_dotenv

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

client = openai.OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

objectifs = [
    "Créer un portfolio",
    "Apprendre React Native",
    "Organiser un déménagement",
    "Lancer une chaîne YouTube",
    "Me remettre au sport après 2 ans d'arrêt",
    "Ecrire un livre",
    "Arriver à courir 2 km",
    "Apprendre à jouer de la guitare",
    "Apprendre à cuisiner des plats végétariens",
    "Apprendre à programmer en Python",
    "Créer un blog"
]

prompt_template = """
Tu es un assistant personnel. À partir de l’objectif donné, génère une liste de 5 à 10 tâches concrètes, claires et actionnables. 
Évite les généralités. Commence chaque tâche par un verbe à l’infinitif.
Formate la réponse sous forme d'une liste de tirets, non numérotée, comme ceci:
"- [tache]
- [tache]
- [tache]
- [tache]
- [tache]
- [tache]"

Objectif : {objectif}
"""

def get_gemini_response(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash-002")
    response = model.generate_content(prompt)
    return response.text.strip()

def get_openrouter_response(prompt, model):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Erreur : {e}"

results = []

openrouter_models = [
    "mistralai/mistral-7b-instruct",
    "meta-llama/llama-2-13b-chat"
]

for objectif in objectifs:
    prompt = prompt_template.format(objectif=objectif)

    try:
        gemini_response = get_gemini_response(prompt)
    except Exception as e:
        gemini_response = f"Erreur : {e}"

    results.append({
        "objectif": objectif,
        "modele": "gemini-1.5-flash-002",
        "reponse": gemini_response
    })

    for model in openrouter_models:
        print(f"⏳ Test modèle {model} sur objectif : {objectif}")
        response = get_openrouter_response(prompt, model)
        results.append({
            "objectif": objectif,
            "modele": model,
            "reponse": response
        })
        time.sleep(1.5)

print("\n✅ Benchmark terminé.")

# Analyse
print("\n📊 Analyse des réponses :")
summary = {}

def analyse_reponse(texte):
    lignes = [l.strip() for l in texte.splitlines() if l.strip()]
    tirets = [l for l in lignes if l.startswith("-")]
    nb_taches = len(tirets)
    format_ok = nb_taches == len(lignes) and 5 <= nb_taches <= 15
    return nb_taches, format_ok

stats_format = {}

for r in results:
    if r["reponse"].startswith("Erreur"):
        continue

    nb_taches, format_ok = analyse_reponse(r["reponse"])
    modele = r["modele"]
    stats_format.setdefault(modele, {"nb_taches": [], "format_ko": 0})
    stats_format[modele]["nb_taches"].append(nb_taches)
    if not format_ok:
        stats_format[modele]["format_ko"] += 1

for modele, infos in stats_format.items():
    taches = infos["nb_taches"]
    moyenne = sum(taches) / len(taches)
    mauvaises_reponses = infos["format_ko"]
    total = len(taches)
    print(f"🔹 {modele}")
    print(f"  - Moyenne de tâches : {moyenne:.1f}")
    print(f"  - Réponses mal formatées ou hors bornes : {mauvaises_reponses} / {total}")

# Résultats disponibles dans la variable "results"


⏳ Test modèle mistralai/mistral-7b-instruct sur objectif : Créer un portfolio
⏳ Test modèle meta-llama/llama-2-13b-chat sur objectif : Créer un portfolio
⏳ Test modèle mistralai/mistral-7b-instruct sur objectif : Apprendre React Native
⏳ Test modèle meta-llama/llama-2-13b-chat sur objectif : Apprendre React Native
⏳ Test modèle mistralai/mistral-7b-instruct sur objectif : Organiser un déménagement
⏳ Test modèle meta-llama/llama-2-13b-chat sur objectif : Organiser un déménagement
⏳ Test modèle mistralai/mistral-7b-instruct sur objectif : Lancer une chaîne YouTube
⏳ Test modèle meta-llama/llama-2-13b-chat sur objectif : Lancer une chaîne YouTube
⏳ Test modèle mistralai/mistral-7b-instruct sur objectif : Me remettre au sport après 2 ans d'arrêt
⏳ Test modèle meta-llama/llama-2-13b-chat sur objectif : Me remettre au sport après 2 ans d'arrêt
⏳ Test modèle mistralai/mistral-7b-instruct sur objectif : Ecrire un livre
⏳ Test modèle meta-llama/llama-2-13b-chat sur objectif : Ecrire un livre
⏳ 

In [22]:
for r in results:
    print(f"\n🎯 Objectif : {r['objectif']}")
    print(f"🧠 Modèle : {r['modele']}")
    print(r['reponse'])



🎯 Objectif : Créer un portfolio
🧠 Modèle : gemini-1.5-flash-002
- Rassembler tous les travaux significatifs (projets, articles, photos, etc.)  sur un support numérique.
- Sélectionner les 5 à 10 meilleurs travaux pour le portfolio en fonction de leur qualité et de leur pertinence.
- Créer un site web simple (ou utiliser une plateforme comme Behance, Clippings.me) pour héberger le portfolio.
- Rédiger une courte description pour chaque projet, mettant en avant les compétences utilisées et les résultats obtenus.
- Choisir des visuels de haute qualité pour accompagner chaque projet.
- Optimiser le portfolio pour les moteurs de recherche (SEO) en incluant des mots-clés pertinents.
- Créer une page "à propos de moi" concise et professionnelle.
- Tester le portfolio sur différents appareils (ordinateur, tablette, smartphone).
- Partager le lien du portfolio sur les réseaux sociaux et autres plateformes pertinentes.
- Demander à des amis ou des professionnels de relire et de donner leur avis